[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_14/MFM_ch14_calibration/MFM_ch14_calibration.ipynb)

# MFM_ch14_calibration

Inference for forecast comparisons and calibration of predictive distributions, from the saved Chapter 14 forecasts: Clark-West tests for nested models, Holm, Romano-Wolf and SPA for the 18 return tests, the non-synchronous-trading check with the BET ETF, exact power of the Kupiec test, PIT histograms, censored Berkowitz tests and equally weighted quantile scores, the DQ test, adaptive conformal recalibration of Chronos-2 quantiles, conditional Giacomini-White tests and the ACF of absolute returns against geometric decay.

*Modelling Financial Markets (MFM), Chapter 14: Deep Learning and Time-Series Foundation Models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_14'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import sys
import json
import time
import subprocess
for pkg, mod in (('arch', 'arch'), ('chronos-forecasting', 'chronos'), ('timesfm', 'timesfm')):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
import torch
import warnings
warnings.filterwarnings('ignore')

## Data, models and tests

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# local copy of data/market when the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

# name -> (symbol, start date); indices and crypto: closing price
SOURCES = {'sp500': ('GSPC.INDX', '1990-01-01'), 'btc': ('BTC-USD.CC', '2014-09-17'), 'bet': ('BET', '1997-09-19')}
LABELS = {'sp500': 'S&P 500', 'btc': 'Bitcoin', 'bet': 'BET'}
ASSETS = list(SOURCES)
TEST_FROM = {'sp500': '2016-01-01', 'btc': '2019-01-01', 'bet': '2016-01-01'}
POST_FROM = '2025-11-03'   # chosen common cutoff: first Monday after the weights of all three foundation models were public
CTX = 512                  # context length (observations) for the foundation models
W = 1000                   # estimation window for HS, GARCH-t, FHS
REFIT = 20                 # GARCH parameters re-estimated every 20 days
A_VAR, A_ES = 0.01, 0.025  # VaR 1%, ES 2.5%
FM_NAMES = {'chronos2': 'Chronos-2', 'bolt': 'Chronos-Bolt', 'timesfm': 'TimesFM-2.5'}
FM_REPO = {'chronos2': 'amazon/chronos-2', 'bolt': 'amazon/chronos-bolt-small',
           'timesfm': 'google/timesfm-2.5-200m-pytorch'}
C2_LEVELS = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
             0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
DEC_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


# =============================================================================
# DATE
# =============================================================================
def read_market(symbol):
    """Read data/market/<SYMBOL>.csv locally or from the GitHub repository."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def load_returns(name):
    """Daily log returns in %, on the series' own calendar."""
    symbol, start = SOURCES[name]
    p = read_market(symbol)['close'].loc[start:]
    p = p[p > 0].dropna()
    return (100 * np.log(p).diff()).dropna().rename(LABELS[name])


def load_rv():
    """Daily realised variance of SPY (regular session, 5-minute returns, %^2)."""
    fname = 'intraday/SPY.US_5m.csv'
    path = os.path.join(MARKET_DIR, fname)
    d = pd.read_csv(path if os.path.exists(path) else REPO_RAW + fname, parse_dates=['datetime_utc'])
    d['day'] = d['datetime_utc'].dt.normalize()
    d['r'] = 100 * np.log(d['close']).groupby(d['day']).diff()
    rv = (d['r'] ** 2).groupby(d['day']).sum()
    n = d.groupby('day').size()
    rv = rv[n >= 70]                       # drop shortened trading days
    rv.index = rv.index.tz_localize(None) if rv.index.tz is not None else rv.index
    rv.name = 'RV'
    return rv


def load_spy_daily():
    """Daily close-to-close returns of SPY (%, adjusted price)."""
    p = read_market('SPY.US')['adjusted_close']
    return (100 * np.log(p).diff()).dropna().rename('SPY')


# =============================================================================
# MODELE FUNDATIONALE (zero-shot)
# =============================================================================
_FM = {}


def load_fm(name):
    """Load a foundation model once (CPU)."""
    if name in _FM:
        return _FM[name]
    import torch
    if name in ('chronos2', 'bolt'):
        from chronos import BaseChronosPipeline
        _FM[name] = BaseChronosPipeline.from_pretrained(FM_REPO[name], device_map='cpu', torch_dtype=torch.float32)
    elif name == 'timesfm':
        import timesfm
        m = timesfm.TimesFM_2p5_200M_torch.from_pretrained(FM_REPO[name])
        m.compile(timesfm.ForecastConfig(max_context=1024, max_horizon=128, normalize_inputs=True,
                                         use_continuous_quantile_head=True, force_flip_invariance=True,
                                         infer_is_positive=False, fix_quantile_crossing=True,
                                         per_core_batch_size=128))
        _FM[name] = m
    else:
        raise ValueError(name)
    return _FM[name]


def fm_forecast(name, contexts, batch=256):
    """One-day-ahead forecast for a list of contexts (1D np.array).
    Returns (levels, Q, point): Q = n x len(levels) matrix of quantiles, point = point forecast
    (the median for all three models: TimesFM-2.5 returns channel 5 of its output, the median)."""
    import torch
    m = load_fm(name)
    Qs, P = [], []
    for i in range(0, len(contexts), batch):
        cb = [np.asarray(c, dtype='float32') for c in contexts[i:i + batch]]
        if name == 'chronos2':
            q, _ = m.predict_quantiles([torch.tensor(c)[None, :] for c in cb], prediction_length=1,
                                       quantile_levels=C2_LEVELS)
            Q = np.stack([qi[0, 0].numpy() for qi in q])
            Qs.append(Q)
            P.append(Q[:, C2_LEVELS.index(0.5)])
        elif name == 'bolt':
            L = max(len(c) for c in cb)
            x = torch.full((len(cb), L), float('nan'))
            for j, c in enumerate(cb):
                x[j, L - len(c):] = torch.tensor(c)
            q, _ = m.predict_quantiles(x, prediction_length=1, quantile_levels=DEC_LEVELS)
            Q = q[:, 0, :].numpy()
            Qs.append(Q)
            P.append(Q[:, DEC_LEVELS.index(0.5)])
        else:
            pf, qf = m.forecast(horizon=1, inputs=cb)
            Qs.append(np.asarray(qf)[:, 0, 1:10])
            P.append(np.asarray(pf)[:, 0])
    levels = C2_LEVELS if name == 'chronos2' else DEC_LEVELS
    return levels, np.vstack(Qs), np.concatenate(P)


def contexts_at(x, origins, ctx=CTX):
    """The context x[t-ctx:t] for each origin t (position in x)."""
    x = np.asarray(x, float)
    return [x[max(0, t - ctx):t] for t in origins]


def quantile_at(levels, Q, u):
    """Quantile of level u by linear interpolation between the available levels; outside the grid:
    the end value (the model gives no more extreme quantiles)."""
    lv = np.asarray(levels)
    return np.array([np.interp(u, lv, row) for row in Q])


def grid_mean(levels, Q, g=lambda v: v):
    """Approximation of E[g(Y)] = integral of g(q_u) over [0, 1], with q_u linear between levels and constant
    outside the grid (understates the tails)."""
    lv = np.r_[0.0, levels, 1.0]
    G = g(np.column_stack([Q[:, :1], Q, Q[:, -1:]]))
    return np.trapezoid(G, lv, axis=1)


def es_from_grid(levels, Q, a=A_ES, n=200):
    """ES_a = -(1/a) * integral_0^a q_u du on the model grid (below the lowest level: constant)."""
    u = (np.arange(n) + 0.5) / n * a
    return -np.mean(np.column_stack([quantile_at(levels, Q, ui) for ui in u]), axis=1)


# =============================================================================
# BASELINE MODELS FOR RETURNS
# =============================================================================
def hist_mean(r, origins):
    x = np.asarray(r, float)
    cs = np.r_[0, np.cumsum(x)]
    return np.array([cs[t] / t for t in origins])


def ar1(r, origins, w=W):
    """AR(1) estimated by OLS on a rolling window of w observations."""
    x = np.asarray(r, float)
    out = []
    for t in origins:
        y = x[max(0, t - w):t]
        X = np.column_stack([np.ones(len(y) - 1), y[:-1]])
        b = np.linalg.lstsq(X, y[1:], rcond=None)[0]
        out.append(b[0] + b[1] * y[-1])
    return np.array(out)


def garch_t(r, origins, w=W, refit=REFIT):
    """GARCH(1,1) with Student-t innovations (constant mean), rolling window of w observations,
    parameters re-estimated every `refit` days. Returns a DataFrame with mu, sigma, nu at each origin
    and the standardised residuals of the window (for FHS)."""
    from arch import arch_model
    x = np.asarray(r, float)
    rows, Z = [], []
    par = None
    for k, t in enumerate(origins):
        y = x[t - w:t]
        if k % refit == 0 or par is None:
            res = arch_model(y, mean='Constant', vol='GARCH', p=1, q=1, dist='t', rescale=False).fit(
                disp='off', show_warning=False)
            par = res.params.values        # mu, omega, alpha, beta, nu
        mu, om, al, be, nu = par
        e = y - mu
        s2 = np.empty(len(y) + 1)
        s2[0] = e.var()
        for i in range(len(y)):
            s2[i + 1] = om + al * e[i] ** 2 + be * s2[i]
        rows.append((mu, np.sqrt(s2[-1]), nu))
        Z.append(e / np.sqrt(s2[:-1]))
    return pd.DataFrame(rows, columns=['mu', 'sigma', 'nu']), Z


def t_std_q(nu, a):
    """Quantile of level a of the standardised Student-t distribution (unit variance)."""
    return stats.t.ppf(a, nu) * np.sqrt((nu - 2) / nu)


def t_std_es(nu, a):
    """ES_a (pozitiv) al distributiei Student-t standardizate: -E[Z | Z <= q_a]."""
    q = stats.t.ppf(a, nu)
    return stats.t.pdf(q, nu) / a * (nu + q ** 2) / (nu - 1) * np.sqrt((nu - 2) / nu)


def emp_var_es(z, a_var=A_VAR, a_es=A_ES):
    """Empirical (positive) VaR and ES of a sample: VaR_a = -q_a, ES_a = -mean of the tail of probability a."""
    z = np.sort(np.asarray(z))
    n = len(z)
    k = int(np.ceil(a_es * n))
    return -np.quantile(z, a_var), -np.quantile(z, a_es), -z[:k].mean()


def risk_table(r, origins, fm_sigma=None):
    """One-day VaR 1%, VaR 2.5%, ES 2.5% for HS, GARCH-t, FHS and (optionally) FHS filtered with the volatility
    of a foundation model: z = r / sigma_FM over the W-day window."""
    x = np.asarray(r, float)
    g, Z = garch_t(r, origins)
    out = {}
    hs = np.array([emp_var_es(x[t - W:t]) for t in origins])
    out['HS'] = hs
    gt = np.column_stack([-(g.mu + g.sigma * t_std_q(g.nu, A_VAR)), -(g.mu + g.sigma * t_std_q(g.nu, A_ES)),
                          -g.mu + g.sigma * t_std_es(g.nu, A_ES)])
    out['GARCH-t'] = gt
    fhs = np.array([emp_var_es(z) for z in Z])
    out['FHS'] = np.column_stack([-g.mu.values + g.sigma.values * fhs[:, 0],
                                  -g.mu.values + g.sigma.values * fhs[:, 1],
                                  -g.mu.values + g.sigma.values * fhs[:, 2]])
    if fm_sigma is not None:
        for nm, sig in fm_sigma.items():   # sig: series aligned with x (NaN where missing)
            s = np.asarray(sig, float)
            rows = []
            for t in origins:
                zz = x[t - W:t] / s[t - W:t]
                rows.append(s[t] * np.array(emp_var_es(zz[np.isfinite(zz)])))
            out[nm] = np.array(rows)
    return {k: pd.DataFrame(v, columns=['VaR1', 'VaR2.5', 'ES2.5']) for k, v in out.items()}


# =============================================================================
# VOLATILITATE REALIZATA: HAR
# =============================================================================
def har_design(y):
    """HAR regressors: yesterday's level, the 5-day average and the 22-day average."""
    s = pd.Series(y)
    d = s.shift(1)
    wk = s.rolling(5).mean().shift(1)
    mo = s.rolling(22).mean().shift(1)
    return np.column_stack([np.ones(len(s)), d, wk, mo])


def har(rv, origins, w=500, log=False):
    """One-day HAR forecast (OLS on a rolling window of w days). log=True: HAR on log RV,
    with the lognormal mean correction exp(s^2/2)."""
    y = np.log(np.asarray(rv, float)) if log else np.asarray(rv, float)
    X = har_design(y)
    out = []
    for t in origins:
        idx = np.arange(max(22, t - w), t)
        b, *_ = np.linalg.lstsq(X[idx], y[idx], rcond=None)
        f = X[t] @ b
        if log:
            s2 = np.var(y[idx] - X[idx] @ b, ddof=4)
            f = np.exp(f + s2 / 2)
        out.append(f)
    return np.array(out)


# =============================================================================
# LSTM (PyTorch, CPU)
# =============================================================================
def make_windows(x, L):
    """Windows of length L as inputs and the next value as target."""
    x = np.asarray(x, float)
    idx = np.arange(L, len(x))
    return np.stack([x[i - L:i] for i in idx]), x[idx]


def train_scaler(x, L, val_frac=0.2):
    """Mean and standard deviation from the training part only: the windows of length L are split in time order and
    the last val_frac of them are kept for early stopping, so their observations do not enter the scaling."""
    x = np.asarray(x, float)
    n = len(x) - L
    nv = int(val_frac * n)
    tr = x[:L + n - nv]
    return tr.mean(), tr.std()


def train_lstm(X, y, hidden=16, epochs=300, lr=3e-3, seed=0, val_frac=0.2, patience=25, weight_decay=1e-4):
    """One-layer LSTM (univariate input, length L) + linear layer; MSE, Adam, early stopping
    on the last val_frac of the sample (time order kept). Returns (forecast function, history)."""
    import torch
    import torch.nn as nn
    torch.manual_seed(seed)
    np.random.seed(seed)

    class Net(nn.Module):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(1, hidden, batch_first=True)
            self.out = nn.Linear(hidden, 1)

        def forward(self, z):
            h, _ = self.lstm(z.unsqueeze(-1))
            return self.out(h[:, -1]).squeeze(-1)

    n = len(y)
    nv = int(val_frac * n)
    Xt, yt = torch.tensor(X[:n - nv], dtype=torch.float32), torch.tensor(y[:n - nv], dtype=torch.float32)
    Xv, yv = torch.tensor(X[n - nv:], dtype=torch.float32), torch.tensor(y[n - nv:], dtype=torch.float32)
    net = Net()
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    best, best_state, wait, hist = np.inf, None, 0, []
    for ep in range(epochs):
        net.train()
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), 256):
            b = perm[i:i + 256]
            opt.zero_grad()
            loss = ((net(Xt[b]) - yt[b]) ** 2).mean()
            loss.backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            lt = float(((net(Xt) - yt) ** 2).mean())
            lv = float(((net(Xv) - yv) ** 2).mean())
        hist.append((lt, lv))
        if lv < best - 1e-7:
            best, wait = lv, 0
            best_state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break
    net.load_state_dict(best_state)
    net.eval()

    def predict(Xn):
        with torch.no_grad():
            return net(torch.tensor(np.asarray(Xn), dtype=torch.float32)).numpy()
    return predict, np.array(hist)


def lstm_param_count(d, h, torch_bias=True):
    """Number of parameters of one LSTM layer: 4 blocks (3 gates + candidate) x (h*d + h*h + h);
    PyTorch uses two bias vectors (b_ih, b_hh), hence 4 x (h*d + h*h + 2h)."""
    return 4 * (h * d + h * h + (2 if torch_bias else 1) * h)


# =============================================================================
# LOSS FUNCTIONS AND TESTS
# =============================================================================
def qlike(y, f):
    """QLIKE = y/f - log(y/f) - 1 (Patton, 2011): ranking-robust for a conditionally unbiased variance proxy."""
    ratio = np.asarray(y) / np.asarray(f)
    return ratio - np.log(ratio) - 1


def pinball(r, q, a):
    """Pinball (quantile) loss of the quantile q of level a of the return r: (a - 1{r < q})(r - q)."""
    r, q = np.asarray(r), np.asarray(q)
    return (a - (r < q)) * (r - q)


def fz0(L, var, es, a=A_ES):
    """FZ0 loss (Patton, Ziegel & Chen, 2019) for losses L = -r, with VaR and ES > 0."""
    L, var, es = map(np.asarray, (L, var, es))
    return (L > var) * (L - var) / (a * es) + var / es + np.log(es) - 1


def kupiec(hits, p):
    """Kupiec POF test: LR_uc ~ chi2(1)."""
    hits = np.asarray(hits, int)
    T, x = len(hits), hits.sum()
    ph = x / T
    ll0 = (T - x) * np.log(1 - p) + x * np.log(p)
    ll1 = (T - x) * np.log(1 - ph) + x * np.log(ph) if 0 < x < T else 0.0
    lr = -2 * (ll0 - ll1)
    return dict(T=int(T), x=int(x), rate=float(ph), LR=float(lr), p=float(stats.chi2.sf(lr, 1)))


def christoffersen(hits, p):
    """Independence test (first-order Markov chain) and conditional-coverage test."""
    h = np.asarray(hits, int)
    a, b = h[:-1], h[1:]
    n00, n01 = np.sum((a == 0) & (b == 0)), np.sum((a == 0) & (b == 1))
    n10, n11 = np.sum((a == 1) & (b == 0)), np.sum((a == 1) & (b == 1))

    def xlogy(n, q):
        return n * np.log(q) if n > 0 else 0.0
    pi01 = n01 / max(n00 + n01, 1)
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)
    l_ind = xlogy(n00, 1 - pi01) + xlogy(n01, pi01) + xlogy(n10, 1 - pi11) + xlogy(n11, pi11)
    l_0 = xlogy(n00 + n10, 1 - pi) + xlogy(n01 + n11, pi)
    lr_ind = -2 * (l_0 - l_ind)
    lr_uc = kupiec(h, p)['LR']
    return dict(n11=int(n11), pi01=float(pi01), pi11=float(pi11), LR_ind=float(lr_ind),
                p_ind=float(stats.chi2.sf(lr_ind, 1)), LR_cc=float(lr_uc + lr_ind),
                p_cc=float(stats.chi2.sf(lr_uc + lr_ind, 2)))


def nw_var(d, lags=None):
    """Long-run variance (Newey-West, Bartlett kernel, autocovariances divided by T)."""
    d = np.asarray(d, float) - np.mean(d)
    T = len(d)
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    v = np.mean(d ** 2)
    for j in range(1, lags + 1):
        v += 2 * (1 - j / (lags + 1)) * np.sum(d[j:] * d[:-j]) / T
    return v


def diebold_mariano(la, lb, lags=None):
    """DM = mean(d) / sqrt(LRV/T), d = la - lb; negative => A has the lower mean loss."""
    d = np.asarray(la, float) - np.asarray(lb, float)
    dm = d.mean() / np.sqrt(nw_var(d, lags) / len(d))
    return dict(dbar=float(d.mean()), DM=float(dm), p=float(2 * stats.norm.sf(abs(dm))))


def mcs(losses, B=1000, block=10, seed=2):
    """Model confidence set (Hansen, Lunde & Nason, 2011), T_max statistic,
    circular block bootstrap. losses: DataFrame T x m. Returns the MCS p-values."""
    rng = np.random.default_rng(seed)
    X = losses.values
    T, m = X.shape
    nb = int(np.ceil(T / block))
    idx = (rng.integers(0, T, (B, nb))[:, :, None] + np.arange(block)) % T
    idx = idx.reshape(B, -1)[:, :T]
    Lbar = X.mean(0)
    Lb = np.stack([X[i].mean(0) for i in idx])
    alive = list(range(m))
    pvals, p_run = {}, 0.0
    while len(alive) > 1:
        a = np.array(alive)
        d = Lbar[a] - Lbar[a].mean()
        db = Lb[:, a] - Lb[:, a].mean(1, keepdims=True)
        se = np.sqrt(np.mean((db - d) ** 2, axis=0))
        t = d / se
        tb = ((db - d) / se).max(1)
        p = float(np.mean(tb >= t.max()))
        p_run = max(p_run, p)
        worst = a[np.argmax(t)]
        pvals[losses.columns[worst]] = p_run
        alive.remove(worst)
    pvals[losses.columns[alive[0]]] = 1.0
    return pd.Series(pvals)[losses.columns]


def r2_oos(y, f, bench=None):
    """Out-of-sample R^2 against a benchmark (default: the zero forecast)."""
    y, f = np.asarray(y), np.asarray(f)
    b = np.zeros_like(y) if bench is None else np.asarray(bench)
    return 1 - np.sum((y - f) ** 2) / np.sum((y - b) ** 2)


def block_bootstrap_ci(stat, arrays, B=2000, block=20, seed=0, level=0.95):
    """Circular block-bootstrap percentile interval for a statistic of several aligned series."""
    rng = np.random.default_rng(seed)
    T = len(arrays[0])
    nb = int(np.ceil(T / block))
    vals = []
    for _ in range(B):
        idx = ((rng.integers(0, T, nb)[:, None] + np.arange(block)) % T).ravel()[:T]
        vals.append(stat(*[np.asarray(a)[idx] for a in arrays]))
    lo, hi = np.quantile(vals, [(1 - level) / 2, 1 - (1 - level) / 2])
    return float(lo), float(hi)


def sign_test(y, f):
    """Share of correct signs and the two-sided binomial p-value (H0: 50%)."""
    y, f = np.asarray(y), np.asarray(f)
    keep = (y != 0) & (f != 0)
    k = int(np.sum(np.sign(y[keep]) == np.sign(f[keep])))
    n = int(keep.sum())
    return dict(rate=k / n, n=n, p=float(stats.binomtest(k, n, 0.5).pvalue))

## Style and helpers

In [ ]:
# Standard MFM style: transparent, English labels, legend at the bottom
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Crimson = '#DC3545'
Teal = '#17A2B8'
Magenta = '#D63384'
Brown = '#795548'
Gray = '#7F7F7F'          # reference lines only
LightGray = '#DADADA'     # bands only
PV_CMAP = LinearSegmentedColormap.from_list('pv', [IDAred, '#F4B6B6', '#FFFFFF', '#BFD8BF', Forest])

RET_MODELS = {'hist': 'Historical mean', 'ar1': 'AR(1)', 'lstm': 'LSTM (5 seeds)',
              'chronos2_point': 'Chronos-2', 'bolt_point': 'Chronos-Bolt', 'timesfm_point': 'TimesFM-2.5'}
RET_COL = {'hist': Amber, 'ar1': Purple, 'lstm': Forest, 'chronos2_point': IDAred, 'bolt_point': Orange,
           'timesfm_point': MainBlue}
RV_MODELS = ['RW', 'HAR', 'logHAR', 'GARCH-t', 'LSTM', 'Chronos-2', 'Chronos-Bolt', 'TimesFM-2.5']
RV_LBL = {'RW': 'Random walk', 'HAR': 'HAR', 'logHAR': 'log-HAR', 'GARCH-t': 'GARCH-t', 'LSTM': 'LSTM',
          'Chronos-2': 'Chronos-2', 'Chronos-Bolt': 'Chronos-Bolt', 'TimesFM-2.5': 'TimesFM-2.5'}
RV_COL = {'RW': Amber, 'HAR': Brown, 'logHAR': Purple, 'GARCH-t': Magenta, 'LSTM': Forest, 'Chronos-2': IDAred,
          'Chronos-Bolt': Orange, 'TimesFM-2.5': MainBlue}
RISK_MODELS = ['HS', 'GARCH-t', 'FHS', 'C2-raw', 'C2-FHS', 'TFM-FHS']
RISK_LBL = {'HS': 'HS', 'GARCH-t': 'GARCH-t', 'FHS': 'FHS', 'C2-raw': 'Chronos-2 quantiles',
            'C2-FHS': 'FHS + Chronos-2 volatility', 'TFM-FHS': 'FHS + TimesFM volatility',
            'bolt-raw': 'Chronos-Bolt quantiles', 'timesfm-raw': 'TimesFM quantiles'}
RISK_COL = {'HS': Teal, 'GARCH-t': MainBlue, 'FHS': Forest, 'C2-raw': IDAred, 'C2-FHS': Orange,
            'TFM-FHS': Purple, 'bolt-raw': Amber, 'timesfm-raw': Magenta}


RES = {}
FC_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/Ch_14/'
FC_DIR = next((d for d in ('.', '..', 'Quantlets/Ch_14', '../Quantlets/Ch_14', '../../Quantlets/Ch_14')
               if os.path.exists(os.path.join(d, 'ch14_rv.csv'))), None)


def save_fig(name):
    """Save the figure as transparent PDF and PNG in OUT_DIR (Drive if chosen above, else the current folder)."""
    out = globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(out, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(out, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def load_csv(name):
    """The forecast table ch14_*.csv (produced by run_forecasts), local or from the repository."""
    path = os.path.join(FC_DIR, name) if FC_DIR else FC_RAW + name
    return pd.read_csv(path, index_col=0, parse_dates=True)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles, labels, ncol=3, y=0.0):
    """One legend for the whole figure, below the panels."""
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

RET_MODELS = {'hist': 'Historical mean', 'ar1': 'AR(1)', 'lstm': 'LSTM (5 seeds)', 'chronos2_point': 'Chronos-2', 'bolt_point': 'Chronos-Bolt', 'timesfm_point': 'TimesFM-2.5'}
RET_COL = {'hist': '#B5853F', 'ar1': '#8E44AD', 'lstm': '#2E7D32', 'chronos2_point': '#CD0000', 'bolt_point': '#E67E22', 'timesfm_point': '#1A3A6E'}
RV_MODELS = ['RW', 'HAR', 'logHAR', 'GARCH-t', 'LSTM', 'Chronos-2', 'Chronos-Bolt', 'TimesFM-2.5']
RV_LBL = {'RW': 'Random walk', 'HAR': 'HAR', 'logHAR': 'log-HAR', 'GARCH-t': 'GARCH-t', 'LSTM': 'LSTM', 'Chronos-2': 'Chronos-2', 'Chronos-Bolt': 'Chronos-Bolt', 'TimesFM-2.5': 'TimesFM-2.5'}
RV_COL = {'RW': '#B5853F', 'HAR': '#795548', 'logHAR': '#8E44AD', 'GARCH-t': '#D63384', 'LSTM': '#2E7D32', 'Chronos-2': '#CD0000', 'Chronos-Bolt': '#E67E22', 'TimesFM-2.5': '#1A3A6E'}
RISK_MODELS = ['HS', 'GARCH-t', 'FHS', 'C2-raw', 'C2-FHS', 'TFM-FHS']
RISK_LBL = {'HS': 'HS', 'GARCH-t': 'GARCH-t', 'FHS': 'FHS', 'C2-raw': 'Chronos-2 quantiles', 'C2-FHS': 'FHS + Chronos-2 volatility', 'TFM-FHS': 'FHS + TimesFM volatility', 'bolt-raw': 'Chronos-Bolt quantiles', 'timesfm-raw': 'TimesFM quantiles'}
RISK_COL = {'HS': '#17A2B8', 'GARCH-t': '#1A3A6E', 'FHS': '#2E7D32', 'C2-raw': '#CD0000', 'C2-FHS': '#E67E22', 'TFM-FHS': '#8E44AD', 'bolt-raw': '#B5853F', 'timesfm-raw': '#D63384'}

## Analysis

In [ ]:
def nw_se(x):
    x = np.asarray(x, float)
    return np.sqrt(nw_var(x) / len(x))


def stationary_idx(T, B, block, rng):
    """Indices of the stationary bootstrap (Politis & Romano): geometric block lengths with mean `block`."""
    idx = np.empty((B, T), int)
    for b in range(B):
        t = rng.integers(T)
        for i in range(T):
            if i > 0 and rng.random() < 1 / block:
                t = rng.integers(T)
            else:
                t = (t + 1) % T if i > 0 else t
            idx[b, i] = t
    return idx


def romano_wolf(D, B=2000, block=20, seed=7):
    """Romano--Wolf stepwise adjusted p-values (studentised statistics) for H0_k: E[d_k] <= 0,
    d_k = benchmark loss - loss of model k (positive = model k better)."""
    D = np.asarray(D, float)
    T, K = D.shape
    rng = np.random.default_rng(seed)
    idx = stationary_idx(T, B, block, rng)
    dbar = D.mean(0)
    se = np.array([nw_se(D[:, k]) for k in range(K)])
    t = dbar / se
    tb = np.stack([(D[i].mean(0) - dbar) / se for i in idx])
    order = np.argsort(-t)
    padj = np.empty(K)
    run = 0.0
    for j, k in enumerate(order):
        rest = order[j:]
        p = np.mean(tb[:, rest].max(1) >= t[k])
        run = max(run, p)
        padj[k] = run
    return t, padj


def spa_consistent(D, B=2000, block=20, seed=11):
    """Hansen's (2005) SPA test, consistent version: H0: no model beats the benchmark.
    d_k = benchmark loss - loss of model k; much worse models (t_k <= -sqrt(2 log log n)) are not
    recentred at zero."""
    D = np.asarray(D, float)
    T, K = D.shape
    rng = np.random.default_rng(seed)
    idx = stationary_idx(T, B, block, rng)
    dbar = D.mean(0)
    se = np.array([nw_se(D[:, k]) for k in range(K)])
    t = dbar / se
    mu_c = np.where(t <= -np.sqrt(2 * np.log(np.log(T))), dbar, 0.0)
    stat = max(t.max(), 0.0)
    tb = np.stack([np.maximum(((D[i].mean(0) - dbar + mu_c) / se).max(), 0.0) for i in idx])
    return float(np.mean(tb >= stat))


def holm(p):
    p = np.asarray(p, float)
    m = len(p)
    o = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for i, k in enumerate(o):
        run = max(run, min(1.0, (m - i) * p[k]))
        adj[k] = run
    return adj


def returns_inference():
    res, allp, keys = {}, [], []
    for a in ASSETS:
        d = load_csv(f'ch14_returns_{a}.csv')
        y = d['y'].values
        r = {}
        Dm = []
        for k in RET:
            f = d[k].values
            l0, l1 = y ** 2, (y - f) ** 2
            dm = (l1 - l0).mean() / nw_se(l1 - l0)
            adj = l0 - (l1 - f ** 2)               # Clark--West: adjusted MSPE, benchmark = zero forecast
            cw = adj.mean() / nw_se(adj)
            # with the zero benchmark adj = 2 y f: Clark--West tests the martingale-difference null E[y_t f_t] = 0,
            # valid for any forecast built from past data (Clark & West, 2006), including fixed pretrained models
            r[k] = dict(dm=float(dm), p_dm1=float(stats.norm.cdf(dm)), cw=float(cw), p_cw=float(stats.norm.sf(cw)))
            Dm.append(l0 - l1)
            allp.append(r[k]['p_dm1'])
            keys.append((a, k))
        Dm = np.column_stack(Dm)
        # mean forecasts from the quantile grids (trapezoid, tails held constant) instead of the medians
        r['r2_gridmean'] = {fm: float(100 * r2_oos(y, grid_mean(lv, d[[f'{fm}_q{u:.2f}' for u in lv]].values)))
                            for fm, lv in (('chronos2', C2_LEVELS), ('timesfm', DEC_LEVELS))}
        t, padj = romano_wolf(Dm)
        for j, k in enumerate(RET):
            r[k]['rw'] = float(padj[j])
        r['spa'] = spa_consistent(Dm)
        # AR(1) on the BET: mean slope and the yearly contribution to the squared-error gain
        if a == 'bet':
            x = load_returns('bet')
            pos = x.index.get_indexer(d.index)
            xv = x.values
            b1 = [np.linalg.lstsq(np.column_stack([np.ones(W - 1), xv[t - W:t - 1]]), xv[t - W + 1:t],
                                  rcond=None)[0][1] for t in pos]
            gy = pd.Series(y ** 2 - (y - d['ar1'].values) ** 2, index=d.index).groupby(d.index.year).sum()
            r['ar1_extra'] = dict(b1_mean=float(np.mean(b1)), b1_min=float(np.min(b1)), b1_max=float(np.max(b1)),
                                  sd_f=float(d['ar1'].std()), corr=float(np.corrcoef(y, d['ar1'])[0, 1]),
                                  gain_2020=float(gy.loc[2020]), gain_2026=float(gy.loc[2026]),
                                  gain_min_year=int(gy.idxmin()), gain_max_year=int(gy.idxmax()),
                                  r2_ex2020_2026=float(100 * r2_oos(y[~d.index.year.isin([2020, 2026])],
                                                                      d['ar1'].values[~d.index.year.isin([2020, 2026])])))
        res[a] = r
    h = holm(allp)
    for (a, k), v in zip(keys, h):
        res[a][k]['holm'] = float(v)
    res['n_tests'] = len(allp)
    res['bonf'] = 0.05 / len(allp)
    OUT['ret'] = res


def nonsync():
    d = load_csv('ch14_returns_bet.csv')
    bet = read_market('BET')['close']
    etf = read_market('TVBETETF.RO')['adjusted_close']
    spx = load_returns('sp500')
    px = pd.concat([bet.rename('bet'), etf.rename('etf')], axis=1, join='inner').dropna()
    px = px[(px > 0).all(1)]
    rr = 100 * np.log(px).diff().dropna()
    rr = rr.loc[d.index[0]:]

    def ac1(v):
        v = np.asarray(v) - np.mean(v)
        return float(np.sum(v[1:] * v[:-1]) / np.sum(v * v))
    betr = d['y']
    f = d['lstm']
    j = pd.concat([rr, f.rename('f')], axis=1, join='inner').dropna()
    # one-day execution lag: the signal formed at the close of t-1 is applied to the return of day t+1
    j['etf_next'] = j['etf'].shift(-1)
    j2 = j.dropna()
    etf_nz = (j['etf'] != 0).mean()

    def r2(y, ff):
        return float(100 * r2_oos(np.asarray(y), np.asarray(ff)))

    def cwp(y, ff):
        y, ff = np.asarray(y), np.asarray(ff)
        adj = y ** 2 - ((y - ff) ** 2 - ff ** 2)
        c = adj.mean() / nw_se(adj)
        return float(c), float(stats.norm.sf(c))
    OUT['nonsync'] = dict(
        ac1_bet=ac1(betr), ac1_sp=ac1(spx.loc[d.index[0]:]), ac1_etf=ac1(j['etf']), ac1_bet_common=ac1(j['bet']),
        corr_f_lag=float(np.corrcoef(f.values[1:], betr.values[:-1])[0, 1]),
        n=int(len(j)), start=str(j.index[0].date()), zero_etf=float(100 * (1 - etf_nz)),
        r2_bet=r2(j['bet'], j['f']), r2_etf=r2(j['etf'], j['f']), r2_etf_next=r2(j2['etf_next'], j2['f']),
        cw_bet=cwp(j['bet'], j['f']), cw_etf=cwp(j['etf'], j['f']), cw_etf_next=cwp(j2['etf_next'], j2['f']))


def kupiec_power():
    crit = stats.chi2.ppf(0.95, 1)
    p0 = 0.01

    def rej(T):
        x = np.arange(T + 1)
        ph = x / T
        with np.errstate(divide='ignore', invalid='ignore'):
            ll1 = np.where((x > 0) & (x < T), (T - x) * np.log(1 - ph) + x * np.log(np.where(ph > 0, ph, 1)), 0.0)
        ll0 = (T - x) * np.log(1 - p0) + x * np.log(p0)
        return -2 * (ll0 - ll1) > crit

    def power(T, p):
        return float(stats.binom.pmf(np.arange(T + 1), T, p)[rej(T)].sum())
    res = {'p1': 0.0167}
    for T in (220, 1000, 2693):
        res[f'pow_{T}'] = power(T, 0.0167)
        res[f'size_{T}'] = power(T, p0)
        res[f'exp_{T}'] = 0.01 * T
    Ts = np.arange(200, 6001, 1)
    pw = np.array([power(T, 0.0167) for T in Ts])
    res['T80'] = int(Ts[np.argmax(pw >= 0.8)])
    res['T80_stable'] = int(Ts[np.where(pw < 0.8)[0].max() + 1])
    res['pow2_220'] = power(220, 0.02)
    res['T80_p2'] = int(Ts[np.argmax(np.array([power(T, 0.02) for T in Ts]) >= 0.8)])
    OUT['power'] = res


def garch_grid(a, dates):
    """GARCH-t and FHS on the same origins as in ch14_risk_*: PIT and the 21-quantile grid."""
    r = load_returns(a)
    x = r.values
    org = r.index.get_indexer(dates)
    gf, Z = garch_t(x, org)
    y = x[org]
    mu, s, nu = gf.mu.values, gf.sigma.values, gf.nu.values
    z = (y - mu) / s
    u_t = stats.t.cdf(z / np.sqrt((nu - 2) / nu), nu)
    q_t = mu[:, None] + s[:, None] * np.column_stack([t_std_q(nu, u) for u in LV])
    u_f = np.array([np.mean(Zi <= zi) for Zi, zi in zip(Z, z)])
    q_f = mu[:, None] + s[:, None] * np.stack([np.quantile(Zi, LV) for Zi in Z])
    return y, (u_t, q_t), (u_f, q_f)


def pit_grid(y, Q):
    """PIT by linear interpolation between the grid quantiles; NaN outside [q_1%, q_99%] (censored)."""
    u = np.full(len(y), np.nan)
    lo = y < Q[:, 0]
    hi = y > Q[:, -1]
    for i in np.where(~lo & ~hi)[0]:
        qi = np.maximum.accumulate(Q[i])
        u[i] = np.interp(y[i], qi, LV)
    return u, lo, hi


def berkowitz_cens(u, lo, hi):
    """LR for mu = 0, sigma = 1 in z = Phi^{-1}(u), censored below 1% and above 99% (Berkowitz, 2001)."""
    cL, cH = stats.norm.ppf(0.01), stats.norm.ppf(0.99)
    z = stats.norm.ppf(np.clip(u[np.isfinite(u)], 1e-9, 1 - 1e-9))
    nL, nH = int(lo.sum()), int(hi.sum())

    def ll(p):
        m, s = p[0], np.exp(p[1])
        return (np.sum(stats.norm.logpdf(z, m, s)) + nL * stats.norm.logcdf((cL - m) / s)
                + nH * stats.norm.logsf((cH - m) / s))
    o = optimize.minimize(lambda p: -ll(p), [0.0, 0.0], method='Nelder-Mead')
    lr = 2 * (-o.fun - ll([0.0, 0.0]))
    return dict(mu=float(o.x[0]), sigma=float(np.exp(o.x[1])), LR=float(lr), p=float(stats.chi2.sf(lr, 2)),
                tail_lo=float(100 * lo.mean()), tail_hi=float(100 * hi.mean()))


def qscore(y, Q, idx=None):
    """Equally weighted quantile score on the grid, (2/K) sum_k pinball_k (a grid analogue of the CRPS)."""
    idx = np.arange(len(LV)) if idx is None else np.asarray(idx)
    s = np.zeros(len(y))
    for k in idx:
        s += 2 * (((y < Q[:, k]).astype(float) - LV[k]) * (Q[:, k] - y))
    return s / len(idx)


def bins_of(u, lo, hi):
    edges = np.r_[0, LV, 1]
    c = np.zeros(len(edges) - 1)
    c[0] += lo.sum()
    c[-1] += hi.sum()
    uu = u[np.isfinite(u)]
    k = np.clip(np.searchsorted(LV, uu, side='right'), 0, len(LV))
    for kk in k:
        c[kk] += 1
    return c, np.diff(edges)


def calibration():
    res, pits = {}, {}
    centre = [k for k, u in enumerate(LV) if 0.1 - 1e-9 <= u <= 0.9 + 1e-9]
    tails = [k for k, u in enumerate(LV) if u < 0.1 - 1e-9 or u > 0.9 + 1e-9]
    for a in ASSETS:
        d = load_csv(f'ch14_returns_{a}.csv')
        rk = load_csv(f'ch14_risk_{a}.csv')
        dates = rk.index
        d = d.loc[dates]
        Qc = d[[f'chronos2_q{u:.2f}' for u in LV]].values
        y, (_, Qt), (_, Qf) = garch_grid(a, dates)
        assert np.allclose(y, d['y'].values)
        # check: the FHS VaR 1% of the recomputed grid equals the one in ch14_risk_*.csv
        assert np.allclose(-Qf[:, 0], rk['FHS|VaR1'].values, atol=1e-6)
        r = {}
        # one convention for all models: PIT by linear interpolation on the 21-level grid, censored outside
        # [q_1%, q_99%]; a return below q_1% is exactly a VaR 1% breach
        for nm, Q in (('C2', Qc), ('GARCH-t', Qt), ('FHS', Qf)):
            u, lo, hi = pit_grid(y, Q)
            c, w = bins_of(u, lo, hi)
            chi = float(np.sum((c - len(y) * w) ** 2 / (len(y) * w)))
            bk = berkowitz_cens(u, lo, hi)
            qs, qc, qt = qscore(y, Q), qscore(y, Q, centre), qscore(y, Q, tails)
            r[nm] = dict(berk=bk, chi2=chi, p_chi2=float(stats.chi2.sf(chi, len(c) - 1)), qs=float(qs.mean()),
                         qs_c=float(qc.mean()), qs_t=float(qt.mean()),
                         dens=(c / (len(y) * w)).tolist(), _qs=qs, _qc=qc, _qt=qt)
        for nm in ('GARCH-t', 'FHS'):
            for s, key in (('_qs', 'dm'), ('_qc', 'dm_c'), ('_qt', 'dm_t')):
                dd = r['C2'][s] - r[nm][s]
                t = dd.mean() / nw_se(dd)
                r['C2'][f'{key}_{nm}'] = float(t)
                r['C2'][f'p{key}_{nm}'] = float(2 * stats.norm.sf(abs(t)))
        for nm in r:
            for s in ('_qs', '_qc', '_qt'):
                r[nm].pop(s)
        res[a] = r
        pits[a] = {nm: r[nm]['dens'] for nm in r}
        res[a]['n'] = int(len(y))
    OUT['calib'] = res
    # chart: relative PIT density on the grid intervals (1 = perfect calibration)
    fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.5), sharey=True)
    edges = np.r_[0, LV, 1]
    hs = []
    for ax, a in zip(axs, ASSETS):
        for nm, col in (('C2', IDAred), ('FHS', Forest), ('GARCH-t', MainBlue)):
            v = np.r_[pits[a][nm], pits[a][nm][-1]]
            h, = ax.step(edges, v, where='post', color=col, lw=1.2)
            if a == 'sp500':
                hs.append(h)
        ax.axhline(1, color=Gray, lw=0.7, ls='--')
        ax.set_title(LABELS[a], color='black')
        ax.set_xlabel('PIT $u_t = \\hat F_t(r_t)$')
        ax.set_ylim(0, 2.0)
    axs[0].set_ylabel('Observed / expected frequency')
    fig_legend_bottom(fig, hs, ['Chronos-2 quantiles (zero-shot)', 'FHS (GARCH-t filter)', 'GARCH-t'], ncol=3,
                      y=0.04)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    save_fig('ch14_pit')


def dq_test(L, var, a=0.01, lags=4):
    hit = (L > var).astype(float) - a
    T = len(hit)
    X = np.column_stack([np.ones(T - lags)] + [hit[lags - j - 1:T - j - 1] for j in range(lags)] + [var[lags:]])
    yv = hit[lags:]
    b = np.linalg.lstsq(X, yv, rcond=None)[0]
    stat = float(b @ X.T @ X @ b / (a * (1 - a)))
    return stat, float(stats.chi2.sf(stat, X.shape[1]))


def backtests():
    res = {}
    for a in ASSETS:
        d = load_csv(f'ch14_risk_{a}.csv')
        L = -d['y'].values
        res[a] = {}
        for m in ('HS', 'GARCH-t', 'FHS', 'C2-raw', 'C2-FHS', 'TFM-FHS'):
            s, p = dq_test(L, d[f'{m}|VaR1'].values)
            res[a][m] = dict(dq=s, p_dq=p)
    OUT['dq'] = res


def aci(y, q, alpha, gamma=0.001, n_cal=250):
    """Adaptive conformal inference (Gibbs & Candes, 2021) for a lower quantile q_t of level alpha.
    Scores E_i = q_i - y_i on the last n_cal days; corrected quantile q_t - Qhat_t, where Qhat_t is the
    ceil((1 - alpha_t)(n_cal + 1))-th smallest score. Published boundary rule: if alpha_t <= 0 or that rank exceeds
    n_cal, Qhat_t = +inf (the interval is the whole real line: VaR = +inf, no breach possible); if alpha_t >= 1,
    Qhat_t = -inf. Update alpha_{t+1} = alpha_t + gamma (alpha - err_t); gamma = 0: split conformal."""
    T = len(y)
    E = q - y
    at = alpha
    shift = np.full(T, np.nan)
    err = np.zeros(T)
    alphas = np.full(T, np.nan)
    for t in range(n_cal, T):
        cal = np.sort(E[t - n_cal:t])
        k = int(np.ceil((1 - at) * (n_cal + 1)))
        if at <= 0 or k > n_cal:
            Qh = np.inf
        elif at >= 1 or k < 1:
            Qh = -np.inf
        else:
            Qh = cal[k - 1]
        shift[t] = Qh
        alphas[t] = at
        err[t] = float(y[t] < q[t] - Qh)
        at = at + gamma * (alpha - err[t])
    return shift, alphas


def conformal():
    res = {}
    for a in ASSETS:
        rk = load_csv(f'ch14_risk_{a}.csv')
        d = load_csv(f'ch14_returns_{a}.csv').loc[rk.index]
        y = rk['y'].values
        L = -y
        Qc = d[[f'chronos2_q{u:.2f}' for u in LV]].values
        q1 = Qc[:, 0]
        q25 = quantile_at(C2_LEVELS, Qc, 0.025)
        out = {}
        # with gamma = 0.005 (the step of Gibbs & Candes' 10% experiments) the level alpha_t falls below the smallest
        # attainable rank and the published rule returns VaR = +inf: count those days
        s_big, _ = aci(y, q1, 0.01, 0.005)
        n_inf_005 = int(np.isinf(s_big[250:]).sum())
        for nm, g in (('C2-ACI', 0.001), ('C2-SCP', 0.0)):
            s1, al1 = aci(y, q1, 0.01, g)
            s25, _ = aci(y, q25, 0.025, g)
            assert np.isfinite(s1[250:]).all() and np.isfinite(s25[250:]).all(), (a, nm)
            v1 = -(q1 - s1)
            v25 = -(q25 - s25)
            es = np.maximum(rk['C2-raw|ES2.5'].values + s25, v25)
            out[nm] = (v1, v25, es)
            if nm == 'C2-ACI' and a == 'sp500':
                aci_path = pd.DataFrame({'alpha_t': al1, 'VaR_aci': v1, 'VaR_raw': -q1, 'FHS': rk['FHS|VaR1'].values,
                                         'loss': L}, index=rk.index)
        keep = np.arange(250, len(y))
        r = {'n': int(len(keep)), 'start': str(rk.index[keep[0]].date()), 'inf_005': n_inf_005,
             'bound': float((0.99 + 0.001) / (0.001 * len(keep)))}
        Fz = {}
        models = ['FHS', 'GARCH-t', 'C2-raw', 'C2-FHS', 'C2-SCP', 'C2-ACI']
        for m in models:
            if m in out:
                v1, v25, es = (x[keep] for x in out[m])
            else:
                v1, v25, es = (rk[f'{m}|{c}'].values[keep] for c in ('VaR1', 'VaR2.5', 'ES2.5'))
            Lk = L[keep]
            h = (Lk > v1).astype(int)
            k = kupiec(h, 0.01)
            c = christoffersen(h, 0.01)
            dq, pdq = dq_test(Lk, v1)
            Fz[m] = fz0(Lk, v25, np.maximum(es, 1e-6))
            r[m] = dict(x=k['x'], rate=100 * k['rate'], p_uc=k['p'], p_ind=c['p_ind'], p_cc=c['p_cc'], p_dq=pdq,
                        fz0=float(Fz[m].mean()), var=float(v1.mean()), exp=0.01 * len(keep))
        for m in models:
            if m != 'FHS':
                dm = diebold_mariano(Fz[m], Fz['FHS'])
                r[m]['dm'] = dm['DM']
                r[m]['pdm'] = dm['p']
        res[a] = r
        if a == 'sp500':
            path = aci_path
    OUT['conf'] = res
    # chart: adaptive level alpha_t and VaR 1% (S&P 500)
    p = path.iloc[250:]
    fig, axs = plt.subplots(2, 1, figsize=(7.0, 3.6), sharex=True, gridspec_kw=dict(height_ratios=[1, 1.4]))
    h1, = axs[0].plot(p.index, 100 * p['alpha_t'], color=Purple, lw=0.9)
    axs[0].axhline(1, color=Gray, lw=0.7, ls='--')
    axs[0].set_ylabel('$\\alpha_t$ (%)')
    h2, = axs[1].plot(p.index, p['VaR_raw'], color=IDAred, lw=0.7)
    h3, = axs[1].plot(p.index, p['VaR_aci'], color=Orange, lw=0.7)
    h4, = axs[1].plot(p.index, p['FHS'], color=Forest, lw=0.7)
    axs[1].set_yscale('log')
    axs[1].set_ylabel('VaR 1% (%)')
    axs[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    fig_legend_bottom(fig, [h1, h2, h3, h4], ['Adaptive level $\\alpha_t$ (target 1%)', 'Chronos-2 raw quantile',
                                              'Chronos-2 + adaptive conformal', 'FHS'], ncol=2, y=0.06)
    fig.tight_layout(rect=(0, 0.1, 1, 1))
    save_fig('ch14_conformal')


def gw():
    d = load_csv('ch14_rv.csv')
    base = np.asarray(qlike(d['RV'], d['logHAR']))
    res = {}
    for m in ('RW', 'HAR', 'GARCH-t', 'LSTM', 'Chronos-2', 'Chronos-Bolt', 'TimesFM-2.5'):
        dd = np.asarray(qlike(d['RV'], d[m])) - base
        Z = np.column_stack([np.ones(len(dd) - 1), dd[:-1]]) * dd[1:, None]
        T = len(Z)
        zb = Z.mean(0)
        Om = (Z - zb).T @ (Z - zb) / T
        st = float(T * zb @ np.linalg.solve(Om, zb))
        res[m] = dict(gw=st, p=float(stats.chi2.sf(st, 2)))
    OUT['gw'] = res


def memory():
    x = load_returns('sp500').loc['2000':].values
    v = np.abs(x) - np.abs(x).mean()
    acf = {L: float(np.sum(v[L:] * v[:-L]) / np.sum(v * v)) for L in (1, 50, 150)}
    f = (acf[50] / acf[1]) ** (1 / 49)
    OUT['mem'] = dict(acf1=acf[1], acf50=acf[50], acf150=acf[150], f=float(f), bf=float(np.log(f / (1 - f))),
                      half=float(np.log(0.5) / np.log(f)), pred150=float(acf[1] * f ** 149),
                      band=float(1.96 / np.sqrt(len(x))))

## Run

In [ ]:
from scipy import optimize
OUT = {}
LV = np.array(C2_LEVELS)
RET = {'hist': 'Historical mean', 'ar1': 'AR(1)', 'lstm': 'LSTM', 'chronos2_point': 'Chronos-2', 'bolt_point': 'Chronos-Bolt', 'timesfm_point': 'TimesFM-2.5'}
BLOCK, B = 20, 2000
memory()
kupiec_power()
gw()
backtests()
nonsync()
returns_inference()
calibration()
conformal()
print(json.dumps({k: OUT[k] for k in ('mem', 'power', 'gw', 'nonsync')}, indent=1, default=float))

![ch14_pit](./ch14_pit.png)

Output: `ch14_pit.pdf`

![ch14_conformal](./ch14_conformal.png)

Output: `ch14_conformal.pdf`